In [4]:
import time
import os
import pandas as pd
import numpy as np
from gurobipy import *

# 東西數量
sample = 2000
# 亂數
c = np.random.randint(20, high = 100, size = sample) 
A = np.random.randint(2, high = 5, size = sample)
b = 1200
print(c.shape[0], A.shape[0])

# 如果不存在則創建目錄"輸出"
if not os.path.isdir('gurobi_output'): 
    os.mkdir('gurobi_output')

m = Model("Knapsack")
# variables
x = m.addVars(c.shape[0], lb = 0, vtype = GRB.BINARY, name = "x") 
m.update()

obj = sum(c[i] * x[i] for i in range(c.shape[0]))
# or MINIMIZE the objective
m.setObjective(obj, GRB.MAXIMIZE) 
# constraint
m.addConstr( quicksum(A[i] * x[i] for i in range(A.shape[0])) <= b) 
# the current time used as the starting time
t = time.process_time() 
m.optimize()
t_opt = time.process_time()
# finished time minus starting time
print("Optimization time:", t_opt - t, " seconds")
# store the optimization problem as text file. Could inspect the problem
m.write('output/Knapsack.lp')
# list for the output of variable names and values
out = [] 
for v in m.getVars():
    # Append variable name and value to the output list
    out.append([v.varName, v.x]) 
# store as a Pandas dataframe 
df = pd.DataFrame(out) 
# Output as an Excel (or csv) file
df.to_csv('gurobi_output/Knapsack.csv') 
t_rec = time.process_time()
print("Recording output time:", t_rec - t_opt, " seconds")

2000 2000
Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i5-3320M CPU @ 2.60GHz, instruction set [SSE2|AVX]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 1 rows, 2000 columns and 2000 nonzeros
Model fingerprint: 0x49a07bb9
Variable types: 0 continuous, 2000 integer (2000 binary)
Coefficient statistics:
  Matrix range     [2e+00, 4e+00]
  Objective range  [2e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 23245.000000
Presolve removed 0 rows and 1760 columns
Presolve time: 0.01s
Presolved: 1 rows, 240 columns, 240 nonzeros
Variable types: 0 continuous, 240 integer (0 binary)
Found heuristic solution: objective 27660.000000

Root relaxation: objective 4.267300e+04, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent   

In [2]:
from gurobipy import *

# 创建 Gurobi 模型
model = Model("Multi-Dimensional Knapsack")

# 定义物品数量和背包数量
num_items = 5
num_dimensions = 3
num_bins = 2

# 物品的价值和重量
values = [15, 10, 12, 8, 5]
weights = [[4, 2, 3], [5, 3, 3], [7, 5, 6], [2, 8, 1], [6, 7, 4]]

# 背包的容量
bin_capacity = [10, 15]

# 创建变量 x[i, j] 表示是否选择物品 i 放入背包 j
x = {}
for i in range(num_items):
    for j in range(num_bins):
        x[i, j] = model.addVar(vtype=GRB.BINARY, name=f'x_{i}_{j}')

# 添加约束：每个物品最多放入一个背包
for i in range(num_items):
    model.addConstr(quicksum(x[i, j] for j in range(num_bins)) <= 1, name=f'item_{i}_constraint')

# 添加约束：每个背包的容量约束
for j in range(num_bins):
    model.addConstr(quicksum(weights[i][j] * x[i, j] for i in range(num_items)) <= bin_capacity[j], name=f'bin_{j}_constraint')

# 最大化总价值
model.setObjective(quicksum(values[i] * x[i, j] for i in range(num_items) for j in range(num_bins)), GRB.MAXIMIZE)

# 优化模型
model.optimize()

# 打印结果
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for i in range(num_items):
        for j in range(num_bins):
            if x[i, j].x > 0.5:
                print(f"Item {i} is placed in Bin {j}")
else:
    print("No optimal solution found!")

# 释放 Gurobi 模型资源
model.dispose()


Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i5-3320M CPU @ 2.60GHz, instruction set [SSE2|AVX]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 7 rows, 10 columns and 20 nonzeros
Model fingerprint: 0x2acbfb23
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [5e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 45.0000000
Presolve time: 0.00s
Presolved: 7 rows, 10 columns, 20 nonzeros
Variable types: 0 continuous, 10 integer (10 binary)

Root relaxation: objective 5.000000e+01, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   50.00000    0    2   45.00000   50.00000  11.1%     -    0s
H    0 